# GluSnFR Bipolar Cell Waveform Clustering and Depth Analysis

This notebook analyzes ex vivo mouse retina GluSnFR recordings from the bipolar cell layer.

## Assumptions

- 2-photon imaging at 50 Hz
- frame size: 128 × 64 pixels
- one ΔF/F waveform per pixel
- known light stimulus onset
- active-pixel threshold: peak ΔF/F ≥ 0.25
- active response must begin near stimulus onset
- recordings are grouped into 7 depth levels
- target output: 7 waveform clusters corresponding to known bipolar cell types

## Main goals

```text
1. Filter for active stimulus-locked pixels
2. Extract waveform features
3. Cluster active waveforms into 7 groups
4. Match cluster mean waveforms to known/control bipolar-cell templates
5. Test whether cluster identity aligns with retinal depth
```

## Required libraries and purpose

| Library | Purpose |
|---|---|
| `pathlib` | file/folder paths |
| `numpy` | numerical waveform calculations |
| `pandas` | load/save waveform and metadata CSVs |
| `matplotlib` | plotting waveforms and figures |
| `seaborn` | heatmaps and group plots |
| `scipy.signal` | response/peak timing features |
| `scipy.cluster.hierarchy` | hierarchical clustering |
| `scipy.stats` | chi-square and group statistics |
| `sklearn.preprocessing` | waveform scaling |
| `sklearn.decomposition` | PCA visualization |
| `sklearn.metrics` | silhouette score |

Install:

```bash
pip install numpy pandas matplotlib seaborn scipy scikit-learn
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

sns.set_context("notebook")
sns.set_style("whitegrid")

## Step 1 — Input files

Expected folder structure:

```text
glusnfr_bipolar_project/
    data/
        pixel_waveforms.csv
        pixel_metadata.csv
        control_reference_waveforms.csv
        known_cell_type_depths.csv
    outputs/
        tables/
        figures/
```

### `pixel_waveforms.csv`

One row per pixel, one column per timepoint.

```text
pixel_id, t0, t1, t2, ...
```

### `pixel_metadata.csv`

```text
pixel_id, depth_id, depth_um, x, y, recording_id
```

Only `pixel_id` and `depth_id` are required.

### `control_reference_waveforms.csv`

Known/control bipolar-cell waveform templates.

```text
cell_type, t0, t1, t2, ...
```

### `known_cell_type_depths.csv`

Known bipolar-cell depth identity.

```text
cell_type, expected_depth_id
```

In [ ]:
project_dir = Path("../glusnfr_bipolar_project")

data_dir = project_dir / "data"
output_dir = project_dir / "outputs"
table_dir = output_dir / "tables"
fig_dir = output_dir / "figures"

for folder in [output_dir, table_dir, fig_dir]:
    folder.mkdir(parents=True, exist_ok=True)

waveform_csv = data_dir / "pixel_waveforms.csv"
metadata_csv = data_dir / "pixel_metadata.csv"
reference_csv = data_dir / "control_reference_waveforms.csv"
known_depth_csv = data_dir / "known_cell_type_depths.csv"

print("Waveforms:", waveform_csv)
print("Metadata:", metadata_csv)
print("Reference templates:", reference_csv)
print("Known depths:", known_depth_csv)

## Step 2 — Load waveforms and metadata

This creates one merged table containing:

```text
pixel metadata + ΔF/F waveform
```

In [ ]:
waveforms_df = pd.read_csv(waveform_csv)

if "pixel_id" not in waveforms_df.columns:
    waveforms_df.insert(
        0,
        "pixel_id",
        [f"pixel_{i:06d}" for i in range(len(waveforms_df))]
    )

metadata_df = pd.read_csv(metadata_csv)

if "pixel_id" not in metadata_df.columns:
    raise ValueError("pixel_metadata.csv must contain pixel_id.")

if "depth_id" not in metadata_df.columns:
    raise ValueError("pixel_metadata.csv must contain depth_id.")

df = metadata_df.merge(
    waveforms_df,
    on="pixel_id",
    how="inner"
)

time_cols = [c for c in waveforms_df.columns if c != "pixel_id"]

for col in time_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Merged shape:", df.shape)
print("Timepoints:", len(time_cols))
display(df.head())

## Step 3 — Define timing and stimulus onset

Frame rate:

```text
50 Hz = 20 ms per frame
```

Update `STIM_ONSET_FRAME` to match your experiment.

In [ ]:
FRAME_RATE_HZ = 50
FRAME_INTERVAL_SEC = 1 / FRAME_RATE_HZ

n_timepoints = len(time_cols)
time_sec = np.arange(n_timepoints) * FRAME_INTERVAL_SEC

STIM_ONSET_FRAME = 10

BASELINE_START_FRAME = 0
BASELINE_END_FRAME = STIM_ONSET_FRAME

RESPONSE_START_FRAME = STIM_ONSET_FRAME
RESPONSE_END_FRAME = min(
    n_timepoints,
    STIM_ONSET_FRAME + int(2.0 * FRAME_RATE_HZ)
)

STIM_ONSET_SEC = STIM_ONSET_FRAME * FRAME_INTERVAL_SEC

print("Stimulus onset frame:", STIM_ONSET_FRAME)
print("Response window:", RESPONSE_START_FRAME, RESPONSE_END_FRAME)

## Step 4 — Filter active stimulus-locked pixels

Criteria:

```text
peak ΔF/F ≥ 0.25
response onset occurs near light stimulus onset
```

Onset is defined as first post-stimulus frame exceeding:

```text
baseline + 3 × baseline standard deviation
```

In [ ]:
ACTIVE_PEAK_THRESHOLD = 0.25
ONSET_Z_THRESHOLD = 3.0
MAX_ONSET_DELAY_SEC = 0.5
MAX_ONSET_DELAY_FRAMES = int(MAX_ONSET_DELAY_SEC * FRAME_RATE_HZ)

def activity_metrics_for_row(row):
    trace = row[time_cols].values.astype(float)

    baseline = np.nanmean(trace[BASELINE_START_FRAME:BASELINE_END_FRAME])
    baseline_std = np.nanstd(trace[BASELINE_START_FRAME:BASELINE_END_FRAME])

    if np.isnan(baseline_std) or baseline_std == 0:
        baseline_std = 1e-8

    corrected = trace - baseline
    response = corrected[RESPONSE_START_FRAME:RESPONSE_END_FRAME]

    peak_dff = np.nanmax(response)
    peak_rel = np.nanargmax(response)
    peak_frame = RESPONSE_START_FRAME + peak_rel

    threshold = ONSET_Z_THRESHOLD * baseline_std
    onset_frame = np.nan

    for frame in range(RESPONSE_START_FRAME, RESPONSE_END_FRAME):
        if corrected[frame] >= threshold:
            onset_frame = frame
            break

    active_peak = peak_dff >= ACTIVE_PEAK_THRESHOLD
    active_onset = (
        not pd.isna(onset_frame)
        and onset_frame <= STIM_ONSET_FRAME + MAX_ONSET_DELAY_FRAMES
    )

    return pd.Series({
        "baseline": baseline,
        "baseline_std": baseline_std,
        "peak_dff": peak_dff,
        "peak_frame": peak_frame,
        "time_to_peak_sec": (peak_frame - STIM_ONSET_FRAME) * FRAME_INTERVAL_SEC,
        "onset_frame": onset_frame,
        "onset_delay_sec": (
            (onset_frame - STIM_ONSET_FRAME) * FRAME_INTERVAL_SEC
            if not pd.isna(onset_frame)
            else np.nan
        ),
        "active_peak": active_peak,
        "active_onset": active_onset,
        "is_active": active_peak and active_onset
    })

activity_metrics = df.apply(activity_metrics_for_row, axis=1)

df_metrics = pd.concat(
    [df[["pixel_id", "depth_id"]].reset_index(drop=True), activity_metrics],
    axis=1
)

active_ids = df_metrics.loc[df_metrics["is_active"], "pixel_id"]

active_df = df[df["pixel_id"].isin(active_ids)].copy()

print("Total pixels:", len(df))
print("Active pixels:", len(active_df))
print("Percent active:", round(100 * len(active_df) / len(df), 2))

df_metrics.to_csv(table_dir / "active_pixel_filter_metrics.csv", index=False)
active_df.to_csv(table_dir / "active_pixel_waveforms.csv", index=False)

display(df_metrics.head())

## Step 5 — Plot active vs inactive waveform examples

Purpose:

```text
verify that active pixels are stimulus-locked and have clear ΔF/F responses
```

In [ ]:
plt.figure(figsize=(10, 5))

for pid in df_metrics.loc[~df_metrics["is_active"], "pixel_id"].head(50):
    trace = df.loc[df["pixel_id"] == pid, time_cols].values.flatten().astype(float)
    plt.plot(time_sec, trace, color="gray", alpha=0.15)

for pid in df_metrics.loc[df_metrics["is_active"], "pixel_id"].head(50):
    trace = df.loc[df["pixel_id"] == pid, time_cols].values.flatten().astype(float)
    plt.plot(time_sec, trace, color="green", alpha=0.25)

plt.axvline(STIM_ONSET_SEC, color="black", linestyle="--", label="Stimulus onset")
plt.xlabel("Time (s)")
plt.ylabel("ΔF/F")
plt.title("Example active and inactive pixel waveforms")
plt.legend()
plt.tight_layout()

out_path = fig_dir / "active_inactive_example_waveforms.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_path)

## Step 6 — Extract waveform features from active pixels

Features are used to interpret clusters.

Features:

```text
peak ΔF/F
AUC
time to peak
FWHM-like width
rise slope
decay slope
duration above 0.25
```

In [ ]:
def extract_waveform_features(row):
    trace = row[time_cols].values.astype(float)

    baseline = np.nanmean(trace[BASELINE_START_FRAME:BASELINE_END_FRAME])
    corrected = trace - baseline

    response = corrected[RESPONSE_START_FRAME:RESPONSE_END_FRAME]
    response_time = time_sec[RESPONSE_START_FRAME:RESPONSE_END_FRAME]

    peak_idx_rel = np.nanargmax(response)
    peak_frame = RESPONSE_START_FRAME + peak_idx_rel
    peak_dff = response[peak_idx_rel]
    time_to_peak = time_sec[peak_frame] - STIM_ONSET_SEC

    auc = np.trapz(response, response_time)

    half_max = peak_dff / 2
    above_half = np.where(response >= half_max)[0]

    fwhm_sec = (
        (above_half[-1] - above_half[0]) * FRAME_INTERVAL_SEC
        if len(above_half) > 1
        else np.nan
    )

    rise_time_sec = max(time_to_peak, FRAME_INTERVAL_SEC)
    rise_slope = peak_dff / rise_time_sec

    decay_segment = response[peak_idx_rel:]

    if len(decay_segment) > 2:
        decay_x = np.arange(len(decay_segment)) * FRAME_INTERVAL_SEC
        decay_slope = np.polyfit(decay_x, decay_segment, 1)[0]
    else:
        decay_slope = np.nan

    duration_above_025 = np.sum(response >= 0.25) * FRAME_INTERVAL_SEC

    return pd.Series({
        "peak_dff": peak_dff,
        "AUC": auc,
        "time_to_peak_sec": time_to_peak,
        "FWHM_sec": fwhm_sec,
        "rise_slope": rise_slope,
        "decay_slope": decay_slope,
        "duration_above_0p25_sec": duration_above_025
    })

feature_rows = active_df.apply(extract_waveform_features, axis=1)

features_df = pd.concat(
    [
        active_df[["pixel_id", "depth_id"]].reset_index(drop=True),
        feature_rows.reset_index(drop=True)
    ],
    axis=1
)

for optional_col in ["depth_um", "x", "y", "recording_id"]:
    if optional_col in active_df.columns:
        features_df[optional_col] = active_df[optional_col].values

features_df.to_csv(table_dir / "active_pixel_waveform_features.csv", index=False)

display(features_df.head())

## Step 7 — Normalize active waveforms for clustering

Hierarchical clustering is performed on waveform shape.

Steps:

```text
baseline subtract
z-score each waveform across time
cluster by correlation distance
```

In [ ]:
X_raw = active_df[time_cols].values.astype(float)

baseline_vals = np.nanmean(
    X_raw[:, BASELINE_START_FRAME:BASELINE_END_FRAME],
    axis=1,
    keepdims=True
)

X_corrected = X_raw - baseline_vals

X_shape = StandardScaler().fit_transform(X_corrected.T).T
X_shape = np.nan_to_num(X_shape)

print("Active waveform matrix:", X_shape.shape)

## Step 8 — Hierarchical clustering into 7 waveform groups

Default:

```text
correlation distance
average linkage
7 clusters
```

In [ ]:
N_CLUSTERS = 7

dist = pdist(X_shape, metric="correlation")
Z = linkage(dist, method="average")

cluster_labels = fcluster(
    Z,
    t=N_CLUSTERS,
    criterion="maxclust"
)

features_df["cluster"] = cluster_labels
active_df["cluster"] = cluster_labels

features_df.to_csv(table_dir / "active_pixel_cluster_assignments.csv", index=False)

print("Cluster counts:")
display(features_df["cluster"].value_counts().sort_index())

plt.figure(figsize=(12, 5))
dendrogram(Z, no_labels=True)
plt.title("Hierarchical clustering dendrogram")
plt.xlabel("Active pixels")
plt.ylabel("Correlation distance")
plt.tight_layout()

out_path = fig_dir / "hierarchical_clustering_dendrogram.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

## Step 9 — Mean waveform per cluster

This is the main waveform identity plot.

In [ ]:
cluster_mean_rows = []

plt.figure(figsize=(10, 6))

for cluster in sorted(features_df["cluster"].unique()):
    idx = features_df["cluster"] == cluster

    mean_waveform = np.nanmean(X_corrected[idx.values], axis=0)

    plt.plot(
        time_sec,
        mean_waveform,
        linewidth=2,
        label=f"Cluster {cluster}, n={idx.sum()}"
    )

    for t, value in zip(time_sec, mean_waveform):
        cluster_mean_rows.append({
            "cluster": cluster,
            "time_sec": t,
            "mean_dff": value
        })

plt.axvline(STIM_ONSET_SEC, color="black", linestyle="--", label="Stimulus")
plt.xlabel("Time (s)")
plt.ylabel("Mean baseline-subtracted ΔF/F")
plt.title("Mean waveform per cluster")
plt.legend()
plt.tight_layout()

out_path = fig_dir / "mean_waveform_per_cluster.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

cluster_means_df = pd.DataFrame(cluster_mean_rows)
cluster_means_df.to_csv(table_dir / "cluster_mean_waveforms.csv", index=False)

## Step 10 — Cluster feature summaries

Purpose:

```text
define what waveform properties characterize each cluster
```

In [ ]:
cluster_feature_summary = (
    features_df
    .groupby("cluster")
    .agg(
        n_pixels=("pixel_id", "count"),
        mean_peak=("peak_dff", "mean"),
        mean_AUC=("AUC", "mean"),
        mean_time_to_peak=("time_to_peak_sec", "mean"),
        mean_FWHM=("FWHM_sec", "mean"),
        mean_duration_above_0p25=("duration_above_0p25_sec", "mean"),
        mean_depth_id=("depth_id", "mean")
    )
    .reset_index()
)

cluster_feature_summary.to_csv(table_dir / "cluster_feature_summary.csv", index=False)

display(cluster_feature_summary)

## Step 11 — Match clusters to control/reference bipolar-cell templates

Matching uses waveform correlation:

```text
cluster mean waveform vs known/control template waveform
```

The best identity is the reference cell type with highest correlation.

In [ ]:
reference_df = pd.read_csv(reference_csv)

if "cell_type" not in reference_df.columns:
    raise ValueError("control_reference_waveforms.csv must contain cell_type.")

reference_time_cols = [c for c in reference_df.columns if c != "cell_type"]

for col in reference_time_cols:
    reference_df[col] = pd.to_numeric(reference_df[col], errors="coerce")

match_rows = []

for cluster in sorted(features_df["cluster"].unique()):
    idx = features_df["cluster"] == cluster
    cluster_mean = np.nanmean(X_corrected[idx.values], axis=0)

    for _, ref_row in reference_df.iterrows():
        ref = ref_row[reference_time_cols].values.astype(float)

        if len(ref) != len(cluster_mean):
            ref_x = np.linspace(0, 1, len(ref))
            cluster_x = np.linspace(0, 1, len(cluster_mean))
            ref = np.interp(cluster_x, ref_x, ref)

        a = cluster_mean - np.nanmean(cluster_mean[:STIM_ONSET_FRAME])
        b = ref - np.nanmean(ref[:min(STIM_ONSET_FRAME, len(ref))])

        corr = np.nan if np.nanstd(a) == 0 or np.nanstd(b) == 0 else np.corrcoef(a, b)[0, 1]

        match_rows.append({
            "cluster": cluster,
            "cell_type": ref_row["cell_type"],
            "template_correlation": corr
        })

template_match_df = pd.DataFrame(match_rows)

best_matches = (
    template_match_df
    .sort_values(["cluster", "template_correlation"], ascending=[True, False])
    .groupby("cluster")
    .head(1)
    .reset_index(drop=True)
)

template_match_df.to_csv(table_dir / "cluster_to_reference_template_correlations.csv", index=False)
best_matches.to_csv(table_dir / "cluster_best_reference_matches.csv", index=False)

display(best_matches)

## Step 12 — Cluster-reference match heatmap

Purpose:

```text
visualize which clusters match which known bipolar-cell templates
```

In [ ]:
match_heatmap = template_match_df.pivot(
    index="cluster",
    columns="cell_type",
    values="template_correlation"
)

plt.figure(figsize=(10, 6))

sns.heatmap(
    match_heatmap,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    cbar_kws={"label": "Waveform correlation"}
)

plt.title("Cluster-to-reference waveform matching")
plt.ylabel("Cluster")
plt.xlabel("Reference bipolar cell type")
plt.tight_layout()

out_path = fig_dir / "cluster_reference_match_heatmap.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

## Step 13 — Depth distribution within clusters

If clusters represent bipolar cell types, clusters should be enriched at specific depths.

In [ ]:
depth_cluster_table = pd.crosstab(
    features_df["depth_id"],
    features_df["cluster"]
)

depth_cluster_table.to_csv(table_dir / "depth_by_cluster_counts.csv")

display(depth_cluster_table)

plt.figure(figsize=(9, 6))

sns.heatmap(
    depth_cluster_table,
    cmap="viridis",
    annot=True,
    fmt="d"
)

plt.title("Depth by cluster count matrix")
plt.xlabel("Cluster")
plt.ylabel("Depth ID")
plt.tight_layout()

out_path = fig_dir / "depth_by_cluster_count_heatmap.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

chi2, p, dof, expected = stats.chi2_contingency(depth_cluster_table)

depth_cluster_stats = pd.DataFrame([{
    "test": "chi_square_depth_by_cluster",
    "chi2": chi2,
    "p_value": p,
    "degrees_of_freedom": dof
}])

depth_cluster_stats.to_csv(table_dir / "depth_cluster_association_statistics.csv", index=False)

display(depth_cluster_stats)

## Step 14 — Validate inferred cell identity by expected depth

Combines:

```text
best waveform template match
observed cluster depth
known expected cell-type depth
```

In [ ]:
known_depth_df = pd.read_csv(known_depth_csv)

if "cell_type" not in known_depth_df.columns:
    raise ValueError("known_cell_type_depths.csv must contain cell_type.")

if "expected_depth_id" not in known_depth_df.columns:
    raise ValueError("known_cell_type_depths.csv must contain expected_depth_id.")

cluster_depth_summary = (
    features_df
    .groupby("cluster")
    .agg(
        observed_mean_depth=("depth_id", "mean"),
        observed_median_depth=("depth_id", "median"),
        n_pixels=("pixel_id", "count")
    )
    .reset_index()
)

cluster_identity_depth = (
    best_matches
    .merge(known_depth_df, on="cell_type", how="left")
    .merge(cluster_depth_summary, on="cluster", how="left")
)

cluster_identity_depth["depth_difference_observed_minus_expected"] = (
    cluster_identity_depth["observed_mean_depth"]
    - cluster_identity_depth["expected_depth_id"]
)

cluster_identity_depth.to_csv(table_dir / "cluster_identity_depth_validation.csv", index=False)

display(cluster_identity_depth)

## Step 15 — Plot expected vs observed depth

Clusters are most convincing if:

```text
observed depth ≈ expected known cell-type depth
```

In [ ]:
plt.figure(figsize=(6, 6))

sns.scatterplot(
    data=cluster_identity_depth,
    x="expected_depth_id",
    y="observed_mean_depth",
    hue="cell_type",
    s=100
)

min_depth = min(
    cluster_identity_depth["expected_depth_id"].min(),
    cluster_identity_depth["observed_mean_depth"].min()
)

max_depth = max(
    cluster_identity_depth["expected_depth_id"].max(),
    cluster_identity_depth["observed_mean_depth"].max()
)

plt.plot(
    [min_depth, max_depth],
    [min_depth, max_depth],
    linestyle="--",
    color="black"
)

plt.xlabel("Expected known cell-type depth ID")
plt.ylabel("Observed mean cluster depth ID")
plt.title("Waveform identity validation by depth")
plt.tight_layout()

out_path = fig_dir / "expected_vs_observed_depth_by_cluster_identity.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

## Step 16 — PCA visualization

PCA is used here only to visualize waveform organization.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_shape)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "cluster": features_df["cluster"].values,
    "depth_id": features_df["depth_id"].values
})

plt.figure(figsize=(7, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="cluster", palette="tab10", s=10, alpha=0.7)
plt.title("PCA of active waveforms by cluster")
plt.tight_layout()
plt.savefig(fig_dir / "pca_active_waveforms_by_cluster.tif", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="depth_id", palette="viridis", s=10, alpha=0.7)
plt.title("PCA of active waveforms by depth")
plt.tight_layout()
plt.savefig(fig_dir / "pca_active_waveforms_by_depth.tif", dpi=300, bbox_inches="tight")
plt.show()

# Final interpretation

A cluster assignment is strongest when all three agree:

```text
1. waveform shape matches a known/control bipolar-cell template
2. pixels from the cluster are enriched at the expected depth
3. cluster features match known response kinetics
```

Example conclusion:

```text
Cluster 3 best matched the BC5 reference waveform and was enriched at depth 5,
consistent with BC5 identity.
```

Important limitation:

Pixels are not independent biological cells. Nearby active pixels may represent the same bouton, process, or neuropil region. For final biological statistics, aggregate by ROI, recording, or animal if metadata are available.